# Machine Learning Project
# Kylle Waldie

# Pokemon Grading Tool

## Web Scraping

### Config

In [1]:
import requests
from bs4 import BeautifulSoup
import time
import re
import csv
import os
from urllib.parse import urljoin

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

BASE_SEARCH_URL = (
    "https://www.ebay.com/sch/i.html"
    "?_nkw=PSA+Pokemon+card"
    "&LH_Sold=1"
    "&LH_Complete=1"
)

IMAGE_DIR = "images"
CSV_FILE = "labels.csv"

os.makedirs(IMAGE_DIR, exist_ok=True)

### Helper Functions

In [2]:
def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=10)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

def extract_grade(title):
    match = re.search(r"PSA\s?(\d+)", title)
    return match.group(1) if match else None

def upgrade_image_url(url):
    return re.sub(r"s-l\d+", "s-l1600", url)

def download_image(url, filename):
    r = requests.get(url, headers=HEADERS, stream=True)
    if r.status_code == 200:
        with open(filename, "wb") as f:
            for chunk in r.iter_content(1024):
                f.write(chunk)

### Scrape Logic

In [3]:
def scrape():
    with open(CSV_FILE, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["filename", "grade", "listing_url"])

        soup = get_soup(BASE_SEARCH_URL)
        items = soup.select(".s-item__link")

        print(f"Found {len(items)} listings")

        for idx, link in enumerate(items):
            listing_url = link.get("href")
            if not listing_url:
                continue

            try:
                listing_soup = get_soup(listing_url)

                title_tag = listing_soup.select_one("#itemTitle")
                if not title_tag:
                    continue

                title = title_tag.get_text(strip=True).replace("Details about", "")
                grade = extract_grade(title)

                if not grade:
                    continue

                img_tag = listing_soup.select_one("#icImg")
                if not img_tag:
                    continue

                img_url = upgrade_image_url(img_tag.get("src"))
                filename = f"psa_{grade}_{idx}.jpg"
                filepath = os.path.join(IMAGE_DIR, filename)

                download_image(img_url, filepath)

                writer.writerow([filename, grade, listing_url])
                print(f"Downloaded PSA {grade} → {filename}")

                time.sleep(2)

            except Exception as e:
                print(f"Skipping listing: {e}")
